# Week 5 — Additional Models: Linear Regression Baseline + XGBoost

**Project:** MaintAI — Predictive Maintenance & RUL Prediction for Turbofan Engines  
**Dataset:** NASA CMAPSS (C-MAPSS FD001–FD004)  

This notebook covers:
1. **Linear Regression** — baseline model (no feature engineering, no tuning)
2. **XGBoost Regressor** — gradient-boosted trees with feature importance
3. **Gradient Boosting (sklearn)** — for comparison
4. Full evaluation: RMSE, MAE, R² for each model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 1. Load Processed Training Data

In [ ]:
import os

# Load all four processed training datasets
data_dir = os.path.join('..', 'Dataset') if os.path.exists(os.path.join('..', 'Dataset')) else 'Dataset'   # adjust path if running from notebooks/

def load_raw(subset_num):
    """Load and preprocess a raw CMAPSS training file."""
    path = os.path.join(data_dir, f'train_FD00{subset_num}.txt')
    cols = ['ID', 'Cycle', 'OpSet1', 'OpSet2', 'OpSet3'] + \
           [f'SensorMeasure{i}' for i in range(1, 22)]
    df = pd.read_csv(path, sep=' ', header=None)
    df.drop(df.columns[[26, 27]], axis=1, inplace=True)   # drop trailing NaN cols
    df.columns = cols
    
    # Compute RUL
    max_cycles = df.groupby('ID')['Cycle'].max().reset_index()
    max_cycles.columns = ['ID', 'MaxCycle']
    df = df.merge(max_cycles, on='ID')
    df['RUL'] = df['MaxCycle'] - df['Cycle']
    df.drop('MaxCycle', axis=1, inplace=True)
    return df

df1 = load_raw(1)
df2 = load_raw(2)
df3 = load_raw(3)
df4 = load_raw(4)

df_all = pd.concat([df1, df2, df3, df4], ignore_index=True)
print(f'Combined dataset shape: {df_all.shape}')
df_all.head()

## 2. Feature Selection

Based on the EDA notebook, we remove sensors with near-zero variance (constant or near-constant):
- SensorMeasure1, 5, 6, 10, 14, 16, 18, 19 (constant across all engines in FD001)

In [ ]:
# Sensors selected from EDA (non-constant, informative)
selected_sensors = ['Cycle',
                    'SensorMeasure2', 'SensorMeasure3', 'SensorMeasure4',
                    'SensorMeasure7', 'SensorMeasure8', 'SensorMeasure11',
                    'SensorMeasure12', 'SensorMeasure13', 'SensorMeasure15',
                    'SensorMeasure17', 'SensorMeasure20', 'SensorMeasure21']

X = df_all[selected_sensors]
y = df_all['RUL']

# Cap RUL at 125 (piece-wise linear degradation model)
y = y.clip(upper=125)

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'\nRUL stats:\n{y.describe()}')

In [ ]:
# Train / test split (80/20, no shuffle to preserve time ordering)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 3. Helper: Evaluation Function

In [ ]:
def evaluate_model(model_name, y_true_train, y_pred_train, y_true_test, y_pred_test):
    """Print RMSE, MAE, R² for both train and test sets."""
    results = {}
    for split, y_t, y_p in [('Train', y_true_train, y_pred_train),
                              ('Test',  y_true_test,  y_pred_test)]:
        rmse = np.sqrt(metrics.mean_squared_error(y_t, y_p))
        mae  = metrics.mean_absolute_error(y_t, y_p)
        r2   = metrics.r2_score(y_t, y_p)
        results[split] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        print(f'{model_name} [{split}]  RMSE: {rmse:.2f}  MAE: {mae:.2f}  R²: {r2:.4f}')
    return results

def plot_predictions(model_name, y_true, y_pred, n=500):
    """Plot actual vs predicted RUL for a sample of n points."""
    idx = np.arange(min(n, len(y_true)))
    plt.figure(figsize=(12, 4))
    plt.plot(idx, np.array(y_true)[:n], label='Actual RUL', color='steelblue')
    plt.plot(idx, np.array(y_pred)[:n], label='Predicted RUL', color='darkorange', alpha=0.8)
    plt.title(f'{model_name} — Actual vs Predicted RUL (first {n} test samples)')
    plt.xlabel('Sample Index'); plt.ylabel('RUL (cycles)')
    plt.legend(); plt.tight_layout(); plt.show()

all_results = {}

## 4. Model 1: Linear Regression (Baseline)

A simple linear model with no hyperparameter tuning serves as the lower-bound baseline.

In [ ]:
# ── Linear Regression ──────────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr_train = lr.predict(X_train_sc)
y_pred_lr_test  = lr.predict(X_test_sc)

print('=== Linear Regression ===')
all_results['Linear Regression'] = evaluate_model(
    'Linear Regression',
    y_train, y_pred_lr_train,
    y_test,  y_pred_lr_test
)
plot_predictions('Linear Regression', y_test.values, y_pred_lr_test)

In [ ]:
# Coefficient analysis
coef_df = pd.DataFrame({'Feature': selected_sensors, 'Coefficient': lr.coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Linear Regression — Feature Coefficients')
plt.tight_layout(); plt.show()
print(coef_df.to_string(index=False))

## 5. Model 2: XGBoost Regressor

XGBoost is a high-performance gradient-boosted tree library, widely used in predictive maintenance.

In [ ]:
# ── XGBoost ────────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators   = 400,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
    n_jobs          = -1,
    verbosity       = 0
)

xgb_model.fit(
    X_train, y_train,
    eval_set    = [(X_test, y_test)],
    verbose     = False
)

y_pred_xgb_train = xgb_model.predict(X_train)
y_pred_xgb_test  = xgb_model.predict(X_test)

print('=== XGBoost Regressor ===')
all_results['XGBoost'] = evaluate_model(
    'XGBoost',
    y_train, y_pred_xgb_train,
    y_test,  y_pred_xgb_test
)
plot_predictions('XGBoost', y_test.values, y_pred_xgb_test)

In [ ]:
# Feature Importance — XGBoost
feat_imp = pd.DataFrame({
    'Feature'   : selected_sensors,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('XGBoost — Feature Importance')
plt.tight_layout(); plt.show()
print(feat_imp.to_string(index=False))

## 6. Model 3: Gradient Boosting (sklearn)

sklearn's GradientBoostingRegressor — slower than XGBoost but useful for comparison.

In [ ]:
# ── Gradient Boosting (sklearn) ────────────────────────────────────────────────
gb_model = GradientBoostingRegressor(
    n_estimators  = 300,
    max_depth     = 5,
    learning_rate = 0.05,
    subsample     = 0.8,
    random_state  = 42
)
gb_model.fit(X_train, y_train)

y_pred_gb_train = gb_model.predict(X_train)
y_pred_gb_test  = gb_model.predict(X_test)

print('=== Gradient Boosting (sklearn) ===')
all_results['Gradient Boosting'] = evaluate_model(
    'Gradient Boosting',
    y_train, y_pred_gb_train,
    y_test,  y_pred_gb_test
)
plot_predictions('Gradient Boosting', y_test.values, y_pred_gb_test)

## 7. Model Comparison Summary

In [ ]:
# Build comparison table
rows = []
for model_name, res in all_results.items():
    rows.append({
        'Model'      : model_name,
        'Train RMSE' : round(res['Train']['RMSE'], 2),
        'Test RMSE'  : round(res['Test']['RMSE'],  2),
        'Train MAE'  : round(res['Train']['MAE'],  2),
        'Test MAE'   : round(res['Test']['MAE'],   2),
        'Train R²'   : round(res['Train']['R2'],   4),
        'Test R²'    : round(res['Test']['R2'],    4),
    })

comparison_df = pd.DataFrame(rows)
print('\n===== MODEL COMPARISON =====')
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = [
    ('Test RMSE', 'RMSE (lower is better)', 'steelblue'),
    ('Test MAE',  'MAE (lower is better)',  'darkorange'),
    ('Test R²',   'R² (higher is better)',  'seagreen'),
]
for ax, (col, title, color) in zip(axes, metrics_to_plot):
    ax.bar(comparison_df['Model'], comparison_df[col], color=color, alpha=0.85)
    ax.set_title(title); ax.set_ylabel(col)
    ax.set_xticklabels(comparison_df['Model'], rotation=15, ha='right')

plt.suptitle('Model Comparison — Week 5 (XGBoost + Linear Regression)', fontsize=13)
plt.tight_layout(); plt.show()

## 8. Save Best Model (XGBoost)

In [ ]:
import joblib, os

model_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else 'models'
os.makedirs(model_dir, exist_ok=True)

# Save XGBoost model and scaler
joblib.dump(xgb_model, os.path.join(model_dir, 'xgboost_rul.pkl'))
joblib.dump(scaler,    os.path.join(model_dir, 'scaler.pkl'))

# Save feature list
import json
with open(os.path.join(model_dir, 'feature_list.json'), 'w') as f:
    json.dump(selected_sensors, f)

print(f'Models saved to: {os.path.abspath(model_dir)}')
print('Files:', os.listdir(model_dir))